# Exploratory Data Analysis: Netflix Movies and TV Shows

**Project:** Exploratory Data Analysis (EDA) using Python, Pandas, and Matplotlib  
**Dataset:** Netflix Movies and TV Shows (Kaggle)  
**Goal:** Understand the dataset's structure and quality, explore content patterns, and communicate evidence-based findings.

> Before running: download `netflix_titles.csv` from the [Kaggle dataset page](https://www.kaggle.com/datasets/shivamb/netflix-shows) and place it in the same folder as this notebook. Do not upload the dataset to GitHub unless you have checked its license and redistribution terms.

This notebook calculates statistics and interpretations from the actual CSV when you run it. Findings are intentionally data-driven rather than guessed in advance.

## 1. Questions to investigate

1. How many rows and columns are present, and what data types are used?
2. Which columns have missing values, and how large is the missing-data problem?
3. What are the most common content types, ratings, genres, and countries?
4. How has the number of titles by release year changed over time?
5. How do movie durations vary, and which ratings are most common?
6. What are the main findings and limitations of this dataset?

In [ ]:
# If needed, install dependencies in a notebook cell with: %pip install pandas matplotlib seaborn
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
DATA_PATH = Path('netflix_titles.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        'Could not find netflix_titles.csv. Download it from '
        'https://www.kaggle.com/datasets/shivamb/netflix-shows and place it beside this notebook.'
    )

df = pd.read_csv(DATA_PATH)
print('Dataset loaded successfully.')


## 2. Structural overview

Check the dataset dimensions, inspect the beginning and end, identify column types, and estimate memory usage before visualizing anything.

In [ ]:
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB')

print('\nColumn data types:')
display(df.dtypes.rename('dtype').to_frame())

print('\nFirst five rows:')
display(df.head())

print('\nLast five rows:')
display(df.tail())

print('\nDetailed structure:')
df.info()


## 3. Data quality and missing values

Count missing values and calculate their percentage for each column. Also check for exact duplicate rows. The chart displays only columns with missing values; if none exist, the notebook reports that instead.

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
}).sort_values('missing_count', ascending=False)
display(missing_summary)

duplicate_count = int(df.duplicated().sum())
print(f'Exact duplicate rows: {duplicate_count:,}')

missing_to_plot = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_percent')
if not missing_to_plot.empty:
    ax = missing_to_plot['missing_percent'].plot(kind='barh')
    ax.set_title('Percentage of Missing Values by Column')
    ax.set_xlabel('Missing values (%)')
    ax.set_ylabel('Column')
    plt.tight_layout()
    plt.show()
    worst = missing_summary.iloc[0]
    print(f"Interpretation: '{missing_summary.index[0]}' has the largest missing-data share ({worst['missing_percent']:.2f}%). Missingness may affect analyses that rely on this field; consider its meaning before filling or dropping values.")
else:
    print('Interpretation: No missing values were found in this dataset.')


## 4. Summary statistics

Numeric summary statistics include mean, median, standard deviation, quartiles, and range. Categorical value counts are shown for key fields when they exist.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
if numeric_cols:
    stats = df[numeric_cols].describe().T
    stats['median'] = df[numeric_cols].median()
    display(stats[['count', 'mean', 'median', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))
else:
    print('No numeric columns were detected by Pandas. Some numeric-looking fields may need parsing.')

for col in ['type', 'rating', 'listed_in', 'country']:
    if col in df.columns:
        print(f'\nTop 10 values for {col}:')
        display(df[col].value_counts(dropna=False).head(10).to_frame('count'))


## 5. Visual analysis

Each chart is paired with a short interpretation generated from the observed data. These interpretations describe associations and counts, not causes.

### Visualization 1 — Are movies or TV shows more common?

This chart compares the number of records by content type.

In [ ]:
if 'type' in df.columns:
    type_counts = df['type'].fillna('Missing').value_counts()
    ax = type_counts.plot(kind='bar')
    ax.set_title('Netflix Titles by Content Type')
    ax.set_xlabel('Content type')
    ax.set_ylabel('Number of titles')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    if len(type_counts):
        top_type = type_counts.index[0]
        share = type_counts.iloc[0] / type_counts.sum() * 100
        print(f'Interpretation: {top_type} is the most common type, representing {share:.1f}% of records in this dataset.')
else:
    print('The expected type column is unavailable.')


### Visualization 2 — How have releases changed over time?

This uses `release_year`, not the date a title was added to Netflix. Recent years can be incomplete in the dataset, so interpret the tail cautiously.

In [ ]:
if 'release_year' in df.columns:
    release_year = pd.to_numeric(df['release_year'], errors='coerce').dropna()
    year_counts = release_year.astype(int).value_counts().sort_index()
    year_counts = year_counts[year_counts.index >= 1980]
    ax = year_counts.plot(kind='line', marker='o', markersize=3)
    ax.set_title('Netflix Titles by Release Year (1980 onward)')
    ax.set_xlabel('Release year')
    ax.set_ylabel('Number of titles in dataset')
    plt.tight_layout()
    plt.show()
    if not year_counts.empty:
        peak_year = int(year_counts.idxmax())
        print(f'Interpretation: The highest count in the plotted period is {int(year_counts.max())} titles for release year {peak_year}. This is the distribution of titles in this dataset, not a complete count of all global releases.')
else:
    print('The expected release_year column is unavailable.')


### Visualization 3 — Which genres appear most often?

A title can belong to multiple listed genres, so split the comma-separated genre field before counting.

In [ ]:
if 'listed_in' in df.columns:
    genre_series = df['listed_in'].dropna().astype(str).str.split(',').explode().str.strip()
    genre_counts = genre_series[genre_series.ne('')].value_counts().head(10).sort_values()
    ax = genre_counts.plot(kind='barh')
    ax.set_title('Top 10 Listed Genres')
    ax.set_xlabel('Number of title-genre entries')
    ax.set_ylabel('Genre')
    plt.tight_layout()
    plt.show()
    if not genre_counts.empty:
        print(f'Interpretation: {genre_counts.idxmax()} is the most frequently listed genre among the top ten. Because titles may have multiple genres, these counts can exceed the number of unique titles.')
else:
    print('The expected listed_in column is unavailable.')


### Visualization 4 — Which content ratings are most common?

Missing ratings are shown explicitly so the chart does not silently discard them.

In [ ]:
if 'rating' in df.columns:
    rating_counts = df['rating'].fillna('Missing').value_counts().head(12).sort_values()
    ax = rating_counts.plot(kind='barh')
    ax.set_title('Most Common Content Ratings')
    ax.set_xlabel('Number of titles')
    ax.set_ylabel('Content rating')
    plt.tight_layout()
    plt.show()
    print(f'Interpretation: {rating_counts.idxmax()} is the most frequent category among the ratings shown. Ratings describe classification labels; they should not be treated as audience scores or quality rankings.')
else:
    print('The expected rating column is unavailable.')


### Visualization 5 — Which countries have the most titles?

Country entries may contain multiple countries for a single title. The chart counts each listed country separately.

In [ ]:
if 'country' in df.columns:
    country_series = df['country'].dropna().astype(str).str.split(',').explode().str.strip()
    country_counts = country_series[country_series.ne('')].value_counts().head(10).sort_values()
    ax = country_counts.plot(kind='barh')
    ax.set_title('Top 10 Countries by Listed Titles')
    ax.set_xlabel('Number of title-country entries')
    ax.set_ylabel('Country')
    plt.tight_layout()
    plt.show()
    if not country_counts.empty:
        print(f'Interpretation: {country_counts.idxmax()} has the largest number of listed title-country entries among the countries shown. Multi-country productions are counted once for each country listed.')
else:
    print('The expected country column is unavailable.')


### Visualization 6 — How are movie durations distributed?

This chart parses duration values such as `90 min` for movies. TV-show seasons are not mixed into the movie-duration distribution.

In [ ]:
if {'type', 'duration'}.issubset(df.columns):
    movies = df[df['type'].eq('Movie')].copy()
    movies['duration_minutes'] = pd.to_numeric(
        movies['duration'].astype('string').str.extract(r'(\d+)')[0], errors='coerce'
    )
    durations = movies['duration_minutes'].dropna()
    if not durations.empty:
        ax = durations.plot(kind='hist', bins=25, edgecolor='black')
        ax.set_title('Distribution of Movie Durations')
        ax.set_xlabel('Duration (minutes)')
        ax.set_ylabel('Number of movies')
        plt.tight_layout()
        plt.show()
        print(f'Interpretation: The median movie duration is {durations.median():.0f} minutes. The histogram shows the spread of durations; unusually short or long titles may warrant checking rather than automatic removal.')
    else:
        print('No movie durations could be parsed from the duration column.')
else:
    print('The type and/or duration columns are unavailable.')


### Optional visualization — Numeric correlations

This heatmap is included only when at least two numeric columns are present. Correlation indicates linear association, not causation.

In [ ]:
numeric_df = df.select_dtypes(include='number')
if numeric_df.shape[1] >= 2:
    corr = numeric_df.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(max(6, len(corr.columns) * 0.8), max(4, len(corr.columns) * 0.6)))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm', aspect='auto')
    ax.set_xticks(range(len(corr.columns)), labels=corr.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(corr.columns)), labels=corr.columns)
    ax.set_title('Correlation Heatmap for Numeric Columns')
    fig.colorbar(im, ax=ax, label='Pearson correlation')
    plt.tight_layout()
    plt.show()
    print('Interpretation: Review the strongest positive and negative coefficients as linear associations. Correlation alone does not establish that one variable causes another.')
else:
    print('Correlation heatmap skipped: fewer than two numeric columns are available.')


## 6. Key findings (generated from the loaded dataset)

The bullets below are calculated when the notebook runs. Review them against the charts and refine the wording with any domain context you discover.

In [ ]:
findings = []
findings.append(f'The dataset contains {df.shape[0]:,} rows and {df.shape[1]:,} columns.')
if 'type' in df.columns and not df['type'].dropna().empty:
    vc = df['type'].value_counts()
    findings.append(f"The most common content type is {vc.index[0]} ({vc.iloc[0]:,} titles; {vc.iloc[0] / vc.sum() * 100:.1f}% of non-missing type values).")
if not missing_summary.empty and missing_summary['missing_count'].max() > 0:
    col = missing_summary['missing_count'].idxmax()
    findings.append(f"The column with the most missing data is '{col}' ({int(missing_summary.loc[col, 'missing_count']):,} missing; {missing_summary.loc[col, 'missing_percent']:.1f}%).")
else:
    findings.append('No missing values were detected in the loaded columns.')
if 'release_year' in df.columns:
    years = pd.to_numeric(df['release_year'], errors='coerce').dropna()
    if not years.empty:
        yvc = years.astype(int).value_counts()
        findings.append(f'The most common release year represented is {int(yvc.idxmax())} ({int(yvc.max()):,} titles).')
if 'listed_in' in df.columns:
    gs = df['listed_in'].dropna().astype(str).str.split(',').explode().str.strip()
    gs = gs[gs.ne('')]
    if not gs.empty:
        findings.append(f"The most frequently listed genre is {gs.value_counts().index[0]} ({int(gs.value_counts().iloc[0]):,} title-genre entries).")
if 'duration' in df.columns and 'type' in df.columns:
    md = df.loc[df['type'].eq('Movie'), 'duration'].astype('string').str.extract(r'(\d+)')[0]
    md = pd.to_numeric(md, errors='coerce').dropna()
    if not md.empty:
        findings.append(f'Movie duration has a median of {md.median():.0f} minutes across {len(md):,} records with parseable durations.')

for i, finding in enumerate(findings[:5], start=1):
    print(f'{i}. {finding}')

display(Markdown('**Interpretation reminder:** These are descriptive findings about the dataset snapshot, not necessarily the entire Netflix catalog or all films and shows worldwide.'))


## 7. Limitations and conclusion

- The data is a snapshot and may not reflect the current Netflix catalog.
- Missing values may be systematic; dropping or imputing them can introduce bias.
- Country and genre fields contain multiple values per title, so exploded counts are entries, not unique titles.
- Release year and date added to Netflix represent different concepts.
- The analysis is descriptive. It does not establish causes or predict future trends.

### Conclusion

This EDA inspects the dataset's structure and quality, summarizes numeric and categorical fields, and explores content types, release years, genres, ratings, countries, and movie durations. Before submitting, rerun all cells from top to bottom, verify that the output matches the data you downloaded, and add any additional interpretation you personally observed.

## Dataset citation

Shivam Bansal, *Netflix Movies and TV Shows*. Kaggle: https://www.kaggle.com/datasets/shivamb/netflix-shows
